In [60]:
import pypsa
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [61]:
# derive PTDF matrix for n0
n0 = pypsa.Network("experiment0.nc")

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, generators, lines, loads, sub_networks


In [62]:
# 1. Extract as a dense NumPy array (restricting to Transmission Lines)
K_array = n0.incidence_matrix(branch_components=["Line"]).toarray()

# 2. Wrap into a labeled pandas DataFrame (Rows = Buses, Columns = Lines)
K_df = pd.DataFrame(
    K_array,
    index=n0.buses.index,
    columns=n0.lines.index
)

print(K_df.T)

name             Spring  Topsham  Gulf  Raymond  Trenton  Chester
name                                                             
Chester-Trenton     0.0      0.0   0.0      0.0     -1.0      1.0
Chester-Gulf        0.0      0.0  -1.0      0.0      0.0      1.0
Trenton-Gulf        0.0      0.0  -1.0      0.0      1.0      0.0
Trenton-Topsham     0.0     -1.0   0.0      0.0      1.0      0.0
Gulf-Spring        -1.0      0.0   1.0      0.0      0.0      0.0
Spring-Topsham      1.0     -1.0   0.0      0.0      0.0      0.0
Spring-Raymond      1.0      0.0   0.0     -1.0      0.0      0.0
Topsham-Raymond     0.0      1.0   0.0     -1.0      0.0      0.0


In [63]:
K = K_array.T

reactances = np.array([13, 18, 9, 15, 18, 9, 12, 14])

Rmat = np.diag(1/reactances)

Bmat = K.T @ Rmat @ K
Bred = Bmat[:5, :5]

theta_red = np.linalg.solve(Bred, np.eye(5))
theta_f = np.vstack( (theta_red, np.zeros(5)) )

pt = Rmat @ K @ theta_f


# 2. Wrap into a labeled pandas DataFrame (Rows = Buses, Columns = Lines)
Hmat = pd.DataFrame(
    pt,
    index=n0.lines.index,
    columns=n0.buses.index[:5]
)

print(Hmat)

name               Spring   Topsham      Gulf   Raymond   Trenton
name                                                             
Chester-Trenton -0.556326 -0.588565 -0.469529 -0.571206 -0.660896
Chester-Gulf    -0.443674 -0.411435 -0.530471 -0.428794 -0.339104
Trenton-Gulf    -0.083766  0.027279 -0.382732 -0.032514  0.276418
Trenton-Topsham -0.472560 -0.615844 -0.086797 -0.538691  0.062687
Gulf-Spring     -0.527440 -0.384156  0.086797 -0.461309 -0.062687
Spring-Topsham   0.351045 -0.285373  0.064478  0.057313 -0.046567
Spring-Raymond   0.121515 -0.098783  0.022319 -0.518622 -0.016119
Topsham-Raymond -0.121515  0.098783 -0.022319 -0.481378  0.016119


In [ ]:
# getting load at each node for time stamp 0 to estimate net generation
# use a copper plate dispatch model


#print(f"Inspecting snapshot: {t0}")

In [ ]:
# 1. don't hardcode wind
# 2. order generators from cheapest to most expensive


t0 = n0.snapshots[0]
loads = n0.loads_t.p.loc[t0] # load by bus
total_load = loads.sum() # total load
loads.index = n0.loads.loc[loads.index, "bus"]

gen_table = n0.generators[["bus", "carrier", "p_nom", "marginal_cost"]]

wind_avail = n0.generators_t.p_max_pu.loc[t0] * gen_table.loc["wind", "p_nom"]

gen_table["p_nom_t"] = gen_table["p_nom"] # make a new column with the time-specific capacity
gen_table.loc["wind", "p_nom_t"] = wind_avail.item()

gens = list(gen_table.index)
amt_used = np.zeros(gen_table.shape[0])
needed = total_load
total = 0

# cumsum column
for source in gen_table.index:
    if needed > 0:
        amt_used[gens.index(source)] = np.minimum(needed, gen_table.loc[source,"p_nom_t"])
    
    needed = needed - amt_used[gens.index(source)]
    
disp = pd.DataFrame({
    "bus": gen_table["bus"],
    "used": np.array(amt_used)
})

disp["load"] = disp["bus"].map(loads)
disp["netg"] = disp["used"] - disp["load"]

netg_vec = disp.set_index("bus")["netg"].loc[Hmat.columns]
#(Hmat * netg_vec).sum(axis=1)






name
load_Spring     48.0
load_Topsham    47.0
load_Gulf       26.0
load_Raymond    18.0
load_Trenton    15.0
load_Chester    14.0
Name: 2025-01-01 05:00:00, dtype: float64

In [ ]:

def get_capacity_t(t):
    n0.loads_t.p_set.loc[t]
    loads.index = n0.loads.loc[loads.index, "bus"]
    capacities = n0.generators[["bus", "p_nom"]] # must be in order of increasing marginal cost
    #wind_avail = n0.generators_t.p_max_pu.loc[t] * gen_table.loc["wind", "p_nom"]

    wind_avail = (n0.generators_t.p_max_pu.loc[t, "wind"] * n0.generators.loc["wind", "p_nom"]
)

    capacities.loc["wind","p_nom"] = wind_avail.item()
    capacities["load"] = capacities["bus"].map(loads)
    return(capacities)

# input: df with load, p_nom, bus; indexed by generator name (ordered least to greatest)
# output: estimated flows on each line

def calc_line_flows(capacity_t, t0):

    amt_used = np.zeros(capacity_t.shape[0])
    needed = capacity_t["load"].sum()
    gens = list(capacity_t.index)
    total = 0


    # cumsum column
    for source in capacity_t.index:
        if needed > 0:
            amt_used[gens.index(source)] = np.minimum(needed, capacity_t.loc[source,"p_nom"])
        
        needed = needed - amt_used[gens.index(source)]

    capacity_t["injection"] = amt_used
    capacity_t = capacity_t.assign(netg = lambda x: x["injection"] - x["load"])

    netg_vec = capacity_t.set_index("bus")["netg"].loc[Hmat.columns]
    return( pd.DataFrame([(Hmat * netg_vec).sum(axis=1)], index = [t0] ) )

all_flows_list = []
for j in n0.snapshots:
    capacities = get_capacity_t(j)
    flow_t = (calc_line_flows(capacities, j))
    all_flows_list.append(flow_t)

flow_df = pd.concat(all_flows_list)

name                 Chester-Trenton  Chester-Gulf  Trenton-Gulf  Trenton-Topsham  Gulf-Spring  \
2025-01-01 05:00:00          2.00631      7.648338     12.398672        54.607638    39.047009   

name                 Spring-Topsham  Spring-Raymond  Topsham-Raymond  
2025-01-01 05:00:00         0.52004        9.872322         8.127678  
name                 Chester-Trenton  Chester-Gulf  Trenton-Gulf  Trenton-Topsham  Gulf-Spring  \
2025-01-01 06:00:00         3.004041      8.779102     13.219033        55.785008    40.998135   

name                 Spring-Topsham  Spring-Raymond  Topsham-Raymond  
2025-01-01 06:00:00       -2.240291        9.455284         9.544716  
name                 Chester-Trenton  Chester-Gulf  Trenton-Gulf  Trenton-Topsham  Gulf-Spring  \
2025-01-01 07:00:00          1.79286      7.897872     13.206057        54.586803    40.103929   

name                 Spring-Topsham  Spring-Raymond  Topsham-Raymond  
2025-01-01 07:00:00       -2.435911        8.849108   